In [ ]:
import scipy.stats as stats

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
import numpy as np
import math

In [ ]:
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr

In [ ]:
import pandas as pd 
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.metrics import average_precision_score
from sklearn.metrics import ndcg_score


In [ ]:
import sys
import psutil 

In [ ]:
X_train_merged_df_men = pd.read_excel('X_train_merged_df_men.xlsx')
X_test_merged_df_men = pd.read_excel('X_test_merged_df_men.xlsx')
X_train_merged_df_men=X_train_merged_df_men.drop(['Unnamed: 0'], axis=1)
X_test_merged_df_men=X_test_merged_df_men.drop(['Unnamed: 0'], axis=1)

In [ ]:
# reverse from libsvm format to dataframe for listnet 8 athletes
import pandas as pd

DESIRED_ORDER = [
    'Competitor', 'Nationality', 'Description', 'Event_Segment', 'Wind', 'Cat', 'Competition_Name', 'Competition_Group', 'Country', 'Key',
    'Age', 'Venue_Familiarity', 'Latitude', 'Longitude', 'Altitude', 'Month', 'Year', 'Temperature', 'Humidity',
    'ApparentTemperature', 'PressureMSL', 'SurfacePressure', 'Total_Participants', 'PB_before', 'SB_before', 'Recent_Avg_5', 'Recent_Best_5', 'Days_Since_Last', 'Weeks_Since_Last', 
    'Max_Rank', 'Relevance', 'Event_ID'
]

# mapping feature index -> column names (excluding Pos and Event_ID)
FEATURE_MAP = {
    1: 'Competitor',
    2: 'Nationality',
    3: 'Description',
    4: 'Event_Segment',
    5: 'Wind',
    6: 'Cat',
    7: 'Competition_Name',
    8: 'Competition_Group',
    9: 'Country',
    10: 'Key',
    11: 'Age',
    12: 'Venue_Familiarity',
    13: 'Latitude',
    14: 'Longitude',
    15: 'Altitude',
    16: 'Month',
    17: 'Year',
    18: 'Temperature',
    19: 'Humidity',
    20: 'ApparentTemperature',
    21: 'PressureMSL',
    22: 'SurfacePressure',
    23: 'Total_Participants',
    24: 'PB_before',
    25: 'SB_before',
    26: 'Recent_Avg_5',
    27: 'Recent_Best_5',
    28: 'Days_Since_Last',
    29: 'Weeks_Since_Last',
    30: 'Max_Rank'
}

def libsvm_swapped_to_original_df(file_path):
    rows = []

    with open(file_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            # ==========================
            # REVERSE THE SWAP
            # ==========================
            Relevance = int(parts[0])                     # relevance = Pos
            Event_ID = int(parts[1].split(":")[1])  # qid = Event_ID
            # ==========================

            # Parse features
            row_data = {
                "Relevance": Relevance,
                "Event_ID": Event_ID
            }

            for feat in parts[2:]:
                idx, val = feat.split(":")
                idx = int(idx)

                if idx in FEATURE_MAP:
                    row_data[FEATURE_MAP[idx]] = float(val)

            rows.append(row_data)

    df = pd.DataFrame(rows)

    # Ensure all columns exist, fill missing with NaN/0
    for col in DESIRED_ORDER:
        if col not in df.columns:
            df[col] = None

    # Reorder columns to original order
    df = df[DESIRED_ORDER]

    return df

restored_men_test_df=libsvm_swapped_to_original_df('test_men.txt')


In [ ]:
restored_men_test_df['Humidity']

In [ ]:
X_test_merged_df_men['Humidity']

In [ ]:
restored_men_test_df['ApparentTemperature']

In [ ]:
X_test_merged_df_men['ApparentTemperature']

In [ ]:
restored_men_test_df

In [ ]:
import numpy as np
import math

def load_scores_with_inf(path):
    scores = []

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            for val in parts:
                fval = float(val)

                # ignore inf or -inf entries (invalid)
                if math.isinf(fval):
                    continue

                scores.append(fval)

    return np.array(scores)

scores_listmle_men = load_scores_with_inf("allrank_listmle_male_combined_tensor_output.txt")


In [ ]:
restored_men_test_df["listmle_men_pred"] = scores_listmle_men

In [ ]:
restored_men_test_df["listmle_men_pred"].describe() 

In [ ]:
restored_men_test_df

In [ ]:
restored_men_test_df.info

In [ ]:
restored_men_test_df.info()

In [ ]:
import numpy as np
import math

def load_scores_with_inf(path):
    scores = []

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            for val in parts:
                fval = float(val)

                # ignore inf or -inf entries (invalid)
                if math.isinf(fval):
                    continue

                scores.append(fval)

    return np.array(scores)

scores_listnet_men = load_scores_with_inf("allrank_listnet_male_combined_tensor_output.txt")


In [ ]:
restored_men_test_df["listnet_men_pred"] = scores_listnet_men

In [ ]:
restored_men_test_df["listnet_men_pred"].describe()

In [ ]:
restored_men_test_df

In [ ]:
ndcg1 = []
ndcg2 = []
ndcg3 = []
ndcg4 = []

for _, group in restored_men_test_df.groupby("Event_ID"):

    y_true = group["Relevance"].values
    y_score = group["listmle_men_pred"].values
    n = len(group)

    ndcg1.append(ndcg_score([y_true], [y_score], k=1))
    if n >= 2:
        ndcg2.append(ndcg_score([y_true], [y_score], k=2))
    if n >= 3:
        ndcg3.append(ndcg_score([y_true], [y_score], k=3))
    if n >= 4:
        ndcg4.append(ndcg_score([y_true], [y_score], k=4))

print(f"Mean NDCG@1: {np.mean(ndcg1):.4f}")
print(f"Mean NDCG@2: {np.mean(ndcg2):.4f}")
print(f"Mean NDCG@3: {np.mean(ndcg3):.4f}")
print(f"Mean NDCG@4: {np.mean(ndcg4):.4f}")

In [ ]:
ndcg1 = []
ndcg2 = []
ndcg3 = []
ndcg4 = []

for _, group in restored_men_test_df.groupby("Event_ID"):

    y_true = group["Relevance"].values
    y_score = group["listnet_men_pred"].values
    n = len(group)

    ndcg1.append(ndcg_score([y_true], [y_score], k=1))
    if n >= 2:
        ndcg2.append(ndcg_score([y_true], [y_score], k=2))
    if n >= 3:
        ndcg3.append(ndcg_score([y_true], [y_score], k=3))
    if n >= 4:
        ndcg4.append(ndcg_score([y_true], [y_score], k=4))

print(f"Mean NDCG@1: {np.mean(ndcg1):.4f}")
print(f"Mean NDCG@2: {np.mean(ndcg2):.4f}")
print(f"Mean NDCG@3: {np.mean(ndcg3):.4f}")
print(f"Mean NDCG@4: {np.mean(ndcg4):.4f}")

In [ ]:
scores = []

for event_id, group in restored_men_test_df.groupby("Event_ID"):
    y_true = group["Relevance"].to_numpy()
    y_pred_event = group["listmle_men_pred"].to_numpy()

    if np.unique(y_true).size < 2 or np.unique(y_pred_event).size < 2:
        continue
    rho, _ = spearmanr(y_true, y_pred_event)
    scores.append(rho)

# print("Mean Spearman:", np.mean(scores))
mean_spearman = np.mean(scores)
std_spearman = np.std(scores)
print(f"Mean spearman: {mean_spearman:.4f}")
print(f"Std spearman: {std_spearman:.4f}")

In [ ]:
scores = []

for event_id, group in restored_men_test_df.groupby("Event_ID"):
    y_true = group["Relevance"].to_numpy()
    y_pred_event = group["listnet_men_pred"].to_numpy()

    if np.unique(y_true).size < 2 or np.unique(y_pred_event).size < 2:
        continue
    rho, _ = spearmanr(y_true, y_pred_event)
    scores.append(rho)

# print("Mean Spearman:", np.mean(scores))
mean_spearman = np.mean(scores)
std_spearman = np.std(scores)
print(f"Mean spearman: {mean_spearman:.4f}")
print(f"Std spearman: {std_spearman:.4f}")

In [ ]:
restored_men_test_df.to_excel("X_test_merged_df_men_allRank.xlsx")

In [ ]:
# reverse from libsvm format to dataframe for listnet 8 athletes
import pandas as pd

DESIRED_ORDER = [
    'Competitor', 'Nationality', 'Description', 'Event_Segment', 'Wind', 'Cat', 'Competition_Name', 'Competition_Group', 'Country', 'Key',
    'Age', 'Venue_Familiarity', 'Latitude', 'Longitude', 'Altitude', 'Month', 'Year', 'Temperature', 'Humidity',
    'ApparentTemperature', 'PressureMSL', 'SurfacePressure', 'Total_Participants', 'PB_before', 'SB_before', 'Recent_Avg_5', 'Recent_Best_5', 'Days_Since_Last', 'Weeks_Since_Last', 
    'Max_Rank', 'Relevance', 'Event_ID'
]

# mapping feature index -> column names (excluding Pos and Event_ID)
FEATURE_MAP = {
    1: 'Competitor',
    2: 'Nationality',
    3: 'Description',
    4: 'Event_Segment',
    5: 'Wind',
    6: 'Cat',
    7: 'Competition_Name',
    8: 'Competition_Group',
    9: 'Country',
    10: 'Key',
    11: 'Age',
    12: 'Venue_Familiarity',
    13: 'Latitude',
    14: 'Longitude',
    15: 'Altitude',
    16: 'Month',
    17: 'Year',
    18: 'Temperature',
    19: 'Humidity',
    20: 'ApparentTemperature',
    21: 'PressureMSL',
    22: 'SurfacePressure',
    23: 'Total_Participants',
    24: 'PB_before',
    25: 'SB_before',
    26: 'Recent_Avg_5',
    27: 'Recent_Best_5',
    28: 'Days_Since_Last',
    29: 'Weeks_Since_Last',
    30: 'Max_Rank'
}

def libsvm_swapped_to_original_df(file_path):
    rows = []

    with open(file_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            # ==========================
            # REVERSE THE SWAP
            # ==========================
            Relevance = int(parts[0])                     # relevance = Pos
            Event_ID = int(parts[1].split(":")[1])  # qid = Event_ID
            # ==========================

            # Parse features
            row_data = {
                "Relevance": Relevance,
                "Event_ID": Event_ID
            }

            for feat in parts[2:]:
                idx, val = feat.split(":")
                idx = int(idx)

                if idx in FEATURE_MAP:
                    row_data[FEATURE_MAP[idx]] = float(val)

            rows.append(row_data)

    df = pd.DataFrame(rows)

    # Ensure all columns exist, fill missing with NaN/0
    for col in DESIRED_ORDER:
        if col not in df.columns:
            df[col] = None

    # Reorder columns to original order
    df = df[DESIRED_ORDER]

    return df

restored_women_test_df=libsvm_swapped_to_original_df('test_women.txt')


In [ ]:
import numpy as np
import math

def load_scores_with_inf(path):
    scores = []

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            for val in parts:
                fval = float(val)

                # ignore inf or -inf entries (invalid)
                if math.isinf(fval):
                    continue

                scores.append(fval)

    return np.array(scores)

scores_listmle_women = load_scores_with_inf("allrank_listmle_women_combined_tensor_output.txt")


In [ ]:
import numpy as np
import math

def load_scores_with_inf(path):
    scores = []

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            for val in parts:
                fval = float(val)

                # ignore inf or -inf entries (invalid)
                if math.isinf(fval):
                    continue

                scores.append(fval)

    return np.array(scores)

scores_listnet_women = load_scores_with_inf("allrank_listnet_women_combined_tensor_output.txt")


In [ ]:
restored_women_test_df["listmle_women_pred"] = scores_listmle_women

In [ ]:
restored_women_test_df["listnet_women_pred"] = scores_listnet_women

In [ ]:
restored_women_test_df

In [ ]:
ndcg1 = []
ndcg2 = []
ndcg3 = []
ndcg4 = []

for _, group in restored_women_test_df.groupby("Event_ID"):

    y_true = group["Relevance"].values
    y_score = group["listmle_women_pred"].values
    n = len(group)

    ndcg1.append(ndcg_score([y_true], [y_score], k=1))
    if n >= 2:
        ndcg2.append(ndcg_score([y_true], [y_score], k=2))
    if n >= 3:
        ndcg3.append(ndcg_score([y_true], [y_score], k=3))
    if n >= 4:
        ndcg4.append(ndcg_score([y_true], [y_score], k=4))

print(f"Mean NDCG@1: {np.mean(ndcg1):.4f}")
print(f"Mean NDCG@2: {np.mean(ndcg2):.4f}")
print(f"Mean NDCG@3: {np.mean(ndcg3):.4f}")
print(f"Mean NDCG@4: {np.mean(ndcg4):.4f}")

In [ ]:
restored_women_test_df["listmle_women_pred"].describe()

In [ ]:
restored_women_test_df["listnet_women_pred"].describe()

In [ ]:
ndcg1 = []
ndcg2 = []
ndcg3 = []
ndcg4 = []

for _, group in restored_women_test_df.groupby("Event_ID"):

    y_true = group["Relevance"].values
    y_score = group["listnet_women_pred"].values
    n = len(group)

    ndcg1.append(ndcg_score([y_true], [y_score], k=1))
    if n >= 2:
        ndcg2.append(ndcg_score([y_true], [y_score], k=2))
    if n >= 3:
        ndcg3.append(ndcg_score([y_true], [y_score], k=3))
    if n >= 4:
        ndcg4.append(ndcg_score([y_true], [y_score], k=4))

print(f"Mean NDCG@1: {np.mean(ndcg1):.4f}")
print(f"Mean NDCG@2: {np.mean(ndcg2):.4f}")
print(f"Mean NDCG@3: {np.mean(ndcg3):.4f}")
print(f"Mean NDCG@4: {np.mean(ndcg4):.4f}")

In [ ]:
scores = []

for event_id, group in restored_women_test_df.groupby("Event_ID"):
    y_true = group["Relevance"].to_numpy()
    y_pred_event = group["listmle_women_pred"].to_numpy()

    if np.unique(y_true).size < 2 or np.unique(y_pred_event).size < 2:
        continue
    rho, _ = spearmanr(y_true, y_pred_event)
    scores.append(rho)

# print("Mean Spearman:", np.mean(scores))
mean_spearman = np.mean(scores)
std_spearman = np.std(scores)
print(f"Mean spearman: {mean_spearman:.4f}")
print(f"Std spearman: {std_spearman:.4f}")

In [ ]:
scores = []

for event_id, group in restored_women_test_df.groupby("Event_ID"):
    y_true = group["Relevance"].to_numpy()
    y_pred_event = group["listnet_women_pred"].to_numpy()

    if np.unique(y_true).size < 2 or np.unique(y_pred_event).size < 2:
        continue
    rho, _ = spearmanr(y_true, y_pred_event)
    scores.append(rho)

# print("Mean Spearman:", np.mean(scores))
mean_spearman = np.mean(scores)
std_spearman = np.std(scores)
print(f"Mean spearman: {mean_spearman:.4f}")
print(f"Std spearman: {std_spearman:.4f}")

In [ ]:
restored_women_test_df.to_excel("X_test_merged_df_women_allRank.xlsx")